# 09.7 - Temperature & Sampling

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Temperature, top-k, top-p, and min-p control the **randomness and diversity** of generated text. Temperature scales logits before softmax; top-k/top-p/min-p filter the vocabulary to a subset before sampling. The same model plus different settings can produce completely different text.

## 2. Why Does This Matter?

Fine control over generation is essential in production: customer service uses low temperature for reliable answers; creative writing uses higher temperature; code generation uses near-zero for correctness. Mis-set sampling parameters are a top cause of poor LLM output.

## 3. Prerequisites

- Unit 09.6 (inference & decoding)

## 4. Learning Objectives

- Explain temperature's effect on the probability distribution
- Implement top-k, top-p, and min-p filtering from scratch
- Understand parameter interactions
- Tune sampling for different tasks

## 5. Mental Model

Temperature is a 'creativity knob': ~0 makes the model conservative, higher makes it adventurous. Filtering (top-k/p/min-p) restricts which candidates are even allowed to be sampled.

```text
logits -> divide by T -> [optional] top-k / top-p / min-p filter -> softmax -> sample
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import torch.nn.functional as F
torch.manual_seed(4)

# A representative next-token distribution (vocab 10)
logits = torch.tensor([2.8, 0.5, 1.9, -0.3, 3.2, 0.1, 2.0, -1.0, 1.2, 0.6])
base = F.softmax(logits, dim=-1)
print("Base probabilities:")
for i, p in enumerate(base):
    print(f"  token {i}: {p.item():.4f}")


Base probabilities:
  token 0: 0.2567
  token 1: 0.0257
  token 2: 0.1044
  token 3: 0.0116
  token 4: 0.3830
  token 5: 0.0173
  token 6: 0.1154
  token 7: 0.0057
  token 8: 0.0518
  token 9: 0.0284


## 7. Temperature Scaling

Compute `softmax(logits / T)`. T near 0 sharpens (peak probability rises); T large flattens (uniform). We also track entropy to quantify randomness.


In [2]:
def softmax_with_temp(lg, T):
    return F.softmax(lg / T, dim=-1)

def entropy(p):
    return float(-(p * torch.log(p + 1e-12)).sum())

print("Temperature effect (peak token = argmax):")
for T in [0.1, 0.3, 0.7, 1.0, 1.5, 3.0]:
    p = softmax_with_temp(logits, T)
    print(f"  T={T:4.1f}: top={int(p.argmax())} p_top={p.max():.3f}  entropy={entropy(p):.3f}")
print("\nAs T->0, p_top->1 (deterministic). As T->inf, distribution->uniform.")


Temperature effect (peak token = argmax):
  T= 0.1: top=4 p_top=0.982  entropy=0.090
  T= 0.3: top=4 p_top=0.771  entropy=0.640
  T= 0.7: top=4 p_top=0.494  entropy=1.377
  T= 1.0: top=4 p_top=0.383  entropy=1.702
  T= 1.5: top=4 p_top=0.284  entropy=1.981
  T= 3.0: top=4 p_top=0.184  entropy=2.212

As T->0, p_top->1 (deterministic). As T->inf, distribution->uniform.


## 8. Tuning the 'Creativity Knob'

Visualize how temperature reshapes the distribution across the vocabulary.


In [3]:
import matplotlib.pyplot as plt
x = list(range(len(logits)))
for T, color in [(0.3, 'red'), (1.0, 'blue'), (3.0, 'green')]:
    p = softmax_with_temp(logits, T)
    plt.plot(x, p.tolist(), marker='o', label=f'T={T}')
plt.xlabel('token')
plt.ylabel('probability')
plt.title('Temperature reshapes the distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('temperature.png', dpi=100)
print("Saved temperature.png")
print("Low T = tall narrow peak; high T = flat spread.")


Saved temperature.png
Low T = tall narrow peak; high T = flat spread.


## 9. Top-k Sampling

Restrict candidates to the k most probable tokens, then renormalize and sample.


In [4]:
def top_k_probs(p, k):
    vals, idx = torch.topk(p, k)
    out = torch.zeros_like(p)
    out[idx] = vals
    return out / out.sum()

for k in [1, 3, 5]:
    pk = top_k_probs(base, k)
    nz = int((pk > 0).sum())
    print(f"top-k={k}: {nz} candidates | draws: {[int(torch.multinomial(pk,1)) for _ in range(8)]}")
print("smaller k = fewer possible tokens = less diversity.")


top-k=1: 1 candidates | draws: [4, 4, 4, 4, 4, 4, 4, 4]
top-k=3: 3 candidates | draws: [0, 4, 4, 4, 4, 0, 4, 6]
top-k=5: 5 candidates | draws: [8, 4, 4, 0, 4, 4, 0, 2]
smaller k = fewer possible tokens = less diversity.


## 10. Top-p (Nucleus) Sampling

Keep the smallest set of tokens whose cumulative probability >= p. Adaptive - picks its own candidate count based on the distribution's shape.


In [5]:
def top_p_probs(p, p_thresh):
    sp, si = torch.sort(p, descending=True)
    cum = torch.cumsum(sp, dim=-1)
    keep = (cum - sp) < p_thresh
    mask = torch.zeros_like(p)
    mask[si[keep]] = 1.0
    out = p * mask
    return out / out.sum()

for pt in [0.3, 0.7, 0.95]:
    pp = top_p_probs(base, pt)
    nz = int((pp > 0).sum())
    print(f"top-p={pt}: keeps {nz} tokens")
print("A flat distribution keeps more tokens; a peaked one keeps fewer.")


top-p=0.3: keeps 1 tokens
top-p=0.7: keeps 3 tokens
top-p=0.95: keeps 7 tokens
A flat distribution keeps more tokens; a peaked one keeps fewer.


## 11. Min-p Sampling

Keep tokens whose probability >= p * (maximum token probability). This drops tokens that are far below the best candidate.


In [6]:
def min_p_probs(p, p_scale):
    p_max = float(p.max())
    threshold = p_scale * p_max
    keep = p >= threshold
    out = p * keep
    return out / out.sum()

print("Base max probability:", f"{base.max():.3f}")
for ps in [0.05, 0.2, 0.5]:
    pm = min_p_probs(base, ps)
    nz = int((pm > 0).sum())
    print(f"min-p={ps}: keeps {nz} tokens (those >= {ps*base.max():.3f})")


Base max probability: 0.383
min-p=0.05: keeps 7 tokens (those >= 0.019)
min-p=0.2: keeps 4 tokens (those >= 0.077)
min-p=0.5: keeps 2 tokens (those >= 0.191)


## 12. Combining Temperature + Top-p

In production you often apply temperature first, then top-p/top-k. Here is the canonical pipeline.


In [7]:
def sample_with_pipeline(logits, temperature=0.8, top_p=0.95, top_k=None):
    p = softmax_with_temp(logits, temperature)
    if top_k:
        p = top_k_probs(p, top_k)
    if top_p:
        p = top_p_probs(p, top_p)
    return int(torch.multinomial(p, 1))

print("10 samples with (T=0.8, top_p=0.95):", [sample_with_pipeline(logits, 0.8, 0.95) for _ in range(10)])
print("Mostly high-prob tokens, occasional variety.")


10 samples with (T=0.8, top_p=0.95):

 [0, 2, 0, 2, 6, 2, 0, 0, 2, 6]
Mostly high-prob tokens, occasional variety.


## 13. Repetition Penalty

A repetition penalty pushes down the logits of already-generated tokens so the model avoids loops.


In [8]:
def apply_repetition_penalty(logits, generated_ids, penalty=1.2):
    lg = logits.clone()
    for t in set(generated_ids):
        if t < lg.numel():
            if lg[t] > 0:
                lg[t] /= penalty
            else:
                lg[t] *= penalty
    return lg

gen = [6, 6, 6]           # token 6 already used
pen = apply_repetition_penalty(logits, gen, penalty=1.4)
print("token 6 base prob:", f"{base[6]:.3f}", "| after penalty:", f"{F.softmax(pen,-1)[6]:.3f}")
print("Repeated token suppressed -> breaks repetition.")


token 6 base prob: 0.115 | after penalty: 0.069
Repeated token suppressed -> breaks repetition.


## 14. Choosing Settings per Task

| Task | Temperature | top-p | Rationale |
|---|---|---|---|
| Customer service | 0.3 | 0.9 | consistent, reliable |
| Code generation | 0.0-0.2 | - | deterministic, correct |
| Chat / Q&A | 0.7 | 0.9 | balanced |
| Creative writing | 0.9-1.0 | 0.95 | diverse, surprising |
| Brainstorming | 1.2+ | 0.9 | exploratory, varied |

## 15. Failure Case & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Always the same output | temperature 0 | raise temperature |
| Too random | temperature too high | lower temperature 0.7-1.0 |
| Sudden quality drops | top-k too small | raise top-k or use top-p |
| Repetitive despite sampling | no repetition penalty | add repetition_penalty |

## 16. Common Mistakes

- High temperature for factual tasks.
- Temperature 0 for creative tasks.
- Ignoring that temperature + top-k + top-p interact (tune together).
- Using one setting for everything.

## 17. When NOT to Use Sampling

- Deterministic output required -> temperature 0 / greedy.
- Strict factual correctness -> low temperature, and validate.

## 18. Challenge

Write a function that maps temperature -> entropy over a grid and report the temperature at which entropy roughly doubles relative to T=1.0.


In [9]:
h1 = entropy(base)
target = 2.0 * h1
for T in torch.linspace(0.1, 5.0, 50):
    h = entropy(softmax_with_temp(logits, float(T)))
    if h >= target:
        print(f"At T={float(T):.1f} entropy {h:.3f} reaches ~2x base entropy {h1:.3f}")
        break
print("Done. Increasing T spreads probability mass and raises entropy.")


Done. Increasing T spreads probability mass and raises entropy.


## 19. Closed-Book Recall

1. What happens to the distribution as temperature -> 0?
2. What is the difference between top-k and top-p?
3. Why might top-p be more adaptive than top-k?
4. How do temperature and repetition penalty interact?

## 20. Teach-Back Questions

Explain to another person:

- Why low temperature reduces randomness.
- How you would tune sampling for a reliable Q&A app.

## 21. Summary

You implemented temperature scaling, top-k, top-p, min-p, a combined pipeline, and a repetition penalty, and mapped settings to tasks.

## 22. Further Experiment

- Add jitter (random noise) to logits and observe sampling robustness.
- Compare top-k vs top-p on a very peaked vs very flat distribution.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
